In [13]:
import pandas as pd
import random
from datetime import datetime, timedelta

In [18]:
import pandas as pd
import random
from datetime import datetime, timedelta

print("Loading and cleaning raw dataset...")
# Make sure your downloaded Kaggle file is in the same folder and named 'crime_data.csv'
raw_data = pd.read_csv('01_District_wise_crimes_committed_IPC_2001_2012.csv')

# Standardize the column names
raw_data.columns = raw_data.columns.str.strip().str.upper()

# Isolate only the Karnataka records
karnataka_data = raw_data[raw_data['STATE/UT'] == 'KARNATAKA'].copy()
print(f"Isolated {len(karnataka_data)} district-year records for Karnataka.")

print("Expanding counts into individual incidents with Incident IDs...")

# Base GPS coordinates for major Karnataka tracking zones
district_coords = {
    'BANGALORE COMMR.': (12.9716, 77.5946),
    'MYSORE': (12.2958, 76.6394),
    'MANGALORE': (12.9141, 74.8560),
    'BELGAUM': (15.8497, 74.4977),
    'HUBLI DHARWAD COMMR.': (15.3647, 75.1240)
}
default_coord = (15.3173, 75.7139) # Center of Karnataka

# Mock names for suspects and victims to enable link analysis later
mock_names = ["Rahul", "Vikram", "Kiran", "Amit", "Anand", "Suresh", "Vijay", "Chethan", "Ravi", "Manoj", "Prakash", "Arun"]

expanded_incidents = []
incident_counter = 10001 # Start ID counter at 10001 for a clean look

# Loop through the rows to expand aggregated numbers into individual rows
for idx, row in karnataka_data.iterrows():
    district = row['DISTRICT']
    year = int(row['YEAR'])

    base_lat, base_lon = district_coords.get(district, default_coord)

    # Selecting the major crime categories
    crime_types = ['MURDER', 'ATTEMPT TO MURDER', 'RAPE', 'KIDNAPPING & ABDUCTION', 'ARSON']

    for crime in crime_types:
        count = int(row[crime])

        for _ in range(count):
            # 1. Generate random date and time
            random_day = random.randint(1, 365)
            hour = random.randint(0, 23)
            minute = random.randint(0, 59)
            date_obj = datetime(year, 1, 1) + timedelta(days=random_day)

            date_str = date_obj.strftime('%Y-%m-%d')
            time_str = f"{hour:02d}:{minute:02d}:00"

            # 2. Add slight variance to coordinates
            lat = base_lat + random.uniform(-0.05, 0.05)
            lon = base_lon + random.uniform(-0.05, 0.05)

            # 3. Assign names randomly
            suspect = random.choice(mock_names) if random.random() > 0.3 else "Unknown"
            victim = random.choice(mock_names)

            # Append with the incident_counter as the first value
            expanded_incidents.append((incident_counter, district, year, crime, date_str, time_str, lat, lon, suspect, victim))

            # Increment the ID for the next row
            incident_counter += 1

# Convert the expanded array into a clean pandas DataFrame with the new ID column
transformed_df = pd.DataFrame(expanded_incidents, columns=[
    'incident_id', 'district', 'year', 'crime_type', 'date', 'time', 'latitude', 'longitude', 'suspect_name', 'victim_name'
])

output_filename = 'karnataka_transformed_crime_data.csv'
transformed_df.to_csv(output_filename, index=False)

print(f"\nSuccess! Brand new granular dataset created.")
print(f"File saved as: '{output_filename}'")
print(f"Total rows generated: {len(transformed_df)}")

Loading and cleaning raw dataset...
Isolated 399 district-year records for Karnataka.
Expanding counts into individual incidents with Incident IDs...

Success! Brand new granular dataset created.
File saved as: 'karnataka_transformed_crime_data.csv'
Total rows generated: 114726


In [20]:
import pandas as pd
import sqlite3

csv_filename = 'karnataka_transformed_crime_data.csv'
print(f"Loading data from '{csv_filename}'...")

# Read the CSV file using pandas
df = pd.read_csv(csv_filename)

db_filename = 'ksp_crime_analytics.db'
print(f"Connecting to SQLite database: '{db_filename}'...")

# Connect to SQLite. If the file doesn't exist, it creates it automatically.
conn = sqlite3.connect(db_filename)

table_name = 'crime_incidents'
print(f"Transferring {len(df)} records to the '{table_name}' table...")

# Write the data to a SQL table.
# 'replace' ensures that if you run this twice, it overwrites the old data instead of duplicating it.
df.to_sql(table_name, conn, if_exists='replace', index=False)

conn.close()
print("Success! CSV data successfully converted and saved into the SQLite DB.")

Loading data from 'karnataka_transformed_crime_data.csv'...
Connecting to SQLite database: 'ksp_crime_analytics.db'...
Transferring 114726 records to the 'crime_incidents' table...
Success! CSV data successfully converted and saved into the SQLite DB.


In [21]:
import pandas as pd
import sqlite3
from sklearn.cluster import KMeans
import warnings

# Ignore minor scikit-learn warnings for cleaner output
warnings.filterwarnings('ignore')

print("Connecting to the database...")
# Connect to your SQLite database file
conn = sqlite3.connect('ksp_crime_analytics.db')

# 1. Pull the data from the database into a pandas DataFrame
# We only need coordinates for the spatial clustering
query = "SELECT incident_id, latitude, longitude, district, crime_type FROM crime_incidents"
crime_df = pd.read_sql_query(query, conn)
print(f"Loaded {len(crime_df)} records for AI clustering.")

# 2. Prepare the data for the AI
# K-Means only understands numbers, so we isolate the GPS coordinates
coords = crime_df[['latitude', 'longitude']]

# 3. Initialize and run the K-Means Machine Learning model
# We tell the AI to look for 5 major crime hotspots across the state (you can adjust this number)
print("Running K-Means Clustering algorithm...")
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

# The AI analyzes the coordinates and assigns a hotspot cluster number (0 to 4) to every crime
crime_df['hotspot_cluster'] = kmeans.fit_predict(coords)

# 4. Save the AI's findings back to the database
# We will create a new table just for the AI insights so we don't mess up the raw data
insights_df = crime_df[['incident_id', 'hotspot_cluster']]
insights_df.to_sql('ai_hotspot_insights', conn, if_exists='replace', index=False)

print("AI Clustering Complete!")
print("Here is a preview of how the AI grouped the hotspots:")
print(crime_df[['district', 'crime_type', 'latitude', 'longitude', 'hotspot_cluster']].head(10))

conn.close()
print("\nHotspot labels successfully saved to the 'ai_hotspot_insights' table in your DB.")

Connecting to the database...
Loaded 114726 records for AI clustering.
Running K-Means Clustering algorithm...
AI Clustering Complete!
Here is a preview of how the AI grouped the hotspots:
   district crime_type   latitude  longitude  hotspot_cluster
0  BAGALKOT     MURDER  15.334973  75.757092                3
1  BAGALKOT     MURDER  15.287616  75.750548                4
2  BAGALKOT     MURDER  15.292064  75.744591                4
3  BAGALKOT     MURDER  15.353093  75.735854                3
4  BAGALKOT     MURDER  15.305843  75.733180                4
5  BAGALKOT     MURDER  15.277224  75.706504                4
6  BAGALKOT     MURDER  15.291202  75.755366                4
7  BAGALKOT     MURDER  15.300236  75.689129                1
8  BAGALKOT     MURDER  15.343059  75.751018                3
9  BAGALKOT     MURDER  15.284749  75.748319                4

Hotspot labels successfully saved to the 'ai_hotspot_insights' table in your DB.


In [22]:
import pandas as pd
import sqlite3
import networkx as nx
import warnings

# Ignore minor warnings for cleaner terminal output
warnings.filterwarnings('ignore')

print("Connecting to the database...")
conn = sqlite3.connect('ksp_crime_analytics.db')

# We only pull cases where the suspect is actually known
# We want to map Suspects -> Victims, and Suspects -> Districts
query = """
    SELECT suspect_name, victim_name, district, crime_type
    FROM crime_incidents
    WHERE suspect_name != 'Unknown'
"""
df = pd.read_sql_query(query, conn)
print(f"Loaded {len(df)} records for Network Analysis.")

print("Building the criminal network web...")
# Initialize an undirected graph
G = nx.Graph()

# Loop through our data and build the connections (edges)
for index, row in df.iterrows():
    # We add prefixes so the system knows the difference between a person and a place
    suspect = f"Suspect: {row['suspect_name']}"
    victim = f"Victim: {row['victim_name']}"
    district = f"Area: {row['district']}"

    # 1. Connect Suspect to Victim
    G.add_edge(suspect, victim, relationship='Targeted')

    # 2. Connect Suspect to District (Operating Area)
    G.add_edge(suspect, district, relationship='Operates_In')

    # Let's add some attributes to the nodes so we can color-code them later
    G.nodes[suspect]['type'] = 'Suspect'
    G.nodes[victim]['type'] = 'Victim'
    G.nodes[district]['type'] = 'Location'

print("Running AI Centrality Algorithms to find key players (Kingpins)...")

# Degree Centrality mathematically calculates how "important" or connected a node is.
# A high score means this suspect is highly active across multiple victims or areas.
centrality = nx.degree_centrality(G)

# Prepare lists to save to our database
nodes_data = []
edges_data = []

# Format the Nodes
for node in G.nodes():
    node_type = G.nodes[node].get('type', 'Unknown')
    score = centrality[node]
    nodes_data.append((node, node_type, score))

# Format the Edges (The lines connecting them)
for source, target, data in G.edges(data=True):
    relationship = data.get('relationship', 'Linked')
    edges_data.append((source, target, relationship))

print("Formatting data for the Dashboard...")
nodes_df = pd.DataFrame(nodes_data, columns=['node_id', 'node_type', 'centrality_score'])
edges_df = pd.DataFrame(edges_data, columns=['source', 'target', 'relationship'])

# Save the network structures back to our SQLite database
nodes_df.to_sql('network_nodes', conn, if_exists='replace', index=False)
edges_df.to_sql('network_edges', conn, if_exists='replace', index=False)

conn.close()

# Sort and print the top 5 most dangerous/connected suspects
top_suspects = nodes_df[nodes_df['node_type'] == 'Suspect'].sort_values(by='centrality_score', ascending=False).head(5)

print("\nNetwork Analysis Complete!")
print("Here are the Top 5 most connected suspects (Kingpins) based on the math:")
print(top_suspects[['node_id', 'centrality_score']])
print("\nSuccess! The spider-web data has been saved to your DB.")

Connecting to the database...
Loaded 80299 records for Network Analysis.
Building the criminal network web...
Running AI Centrality Algorithms to find key players (Kingpins)...
Formatting data for the Dashboard...

Network Analysis Complete!
Here are the Top 5 most connected suspects (Kingpins) based on the math:
           node_id  centrality_score
0  Suspect: Vikram          0.813559
3   Suspect: Rahul          0.813559
6   Suspect: Kiran          0.813559
7   Suspect: Vijay          0.813559
9    Suspect: Amit          0.813559

Success! The spider-web data has been saved to your DB.
